# 1. Introducción

**¿Qué es el EDA?**
El Análisis Exploratorio de Datos (EDA) es una fase fundamental en el proceso de ciencia de datos. Su objetivo es examinar los datos de manera preliminar para descubrir patrones, identificar anomalías, probar hipótesis y comprobar suposiciones utilizando estadísticas resumidas y representaciones gráficas.

**Objetivo del análisis:**
El objetivo principal de este EDA es analizar las emisiones y absorciones de gases de efecto invernadero (GEI) por departamento y sector económico en Colombia, para identificar patrones territoriales y calcular indicadores de contaminación neta.

**Relación con el problema de investigación:**
Este análisis permitirá responder preguntas clave sobre qué departamentos y sectores son los mayores emisores y cuáles presentan mejores balances ambientales, guiando futuras decisiones y modelos predictivos (Machine Learning).

---

### Configuración para Google Colab
Si estás ejecutando este cuaderno en Google Colab, ejecuta la siguiente celda para montar tu Google Drive. Asegúrate de haber subido el archivo `dataset_limpio.csv` a tu Drive. Alternativamente, puedes subir el archivo temporalmente en la barra lateral izquierda.

In [ ]:
# Descomenta las siguientes dos líneas si deseas cargar desde Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Si usas Drive, ajusta la ruta a donde guardaste el dataset, por ejemplo:
# file_path = '/content/drive/MyDrive/dataset_limpio.csv'

# Si subiste el archivo directamente a Colab temporalmente, usa esta ruta:
file_path = '/content/dataset_limpio.csv'


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import missingno as msno
import os
import warnings
warnings.filterwarnings('ignore')

# Directorio para guardar las imágenes en el entorno de Colab
os.makedirs("/content/images/eda", exist_ok=True)

try:
    # Carga del dataset
    df = pd.read_csv(file_path)
    print("Dataset cargado exitosamente.")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo en la ruta {file_path}. Por favor súbelo o ajusta la ruta.")

# Asegurarse de que las variables importantes son numéricas
cols_num = ['emisiones_totales', 'abosorciones_totales', 'emisiones_netas', 'co2', 'ch4', 'n2o', 'co2eq', 'ch4eq', 'n2oeq']
if 'df' in locals():
    for col in cols_num:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')


# 2. Exploración General

En esta sección visualizamos la estructura general del dataset, sus dimensiones, información sobre los tipos de datos y estadísticas descriptivas básicas.

In [ ]:
if 'df' in locals():
    print("Forma del dataset:", df.shape)
    print("\nInformación del dataset:")
    df.info()
    print("\nPrimeros 5 registros:")
    display(df.head())
    print("\nEstadísticas descriptivas:")
    display(df.describe(include='all'))

**Interpretación de Resultados:**
- `df.shape` nos indica el número total de registros y variables disponibles.
- `df.info()` revela el tipo de cada variable (numérica o categórica) y si existen valores nulos.
- `df.describe()` proporciona un resumen estadístico (media, desviación estándar, min, max, cuartiles) para variables numéricas, lo que nos da una idea inicial de su distribución y posibles sesgos.
- La visualización de `df.head()` nos permite confirmar la correcta carga de los datos.

---

# 3. Calidad de Datos

Analizamos valores nulos, duplicados y la consistencia general del conjunto de datos.

In [ ]:
if 'df' in locals():
    # Valores nulos
    nulos = df.isnull().sum()
    nulos_porcentaje = (nulos / len(df)) * 100
    df_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': nulos_porcentaje}).sort_values(by='Porcentaje (%)', ascending=False)
    display(df_nulos[df_nulos['Nulos'] > 0])

    # Visualización de nulos
    plt.figure(figsize=(10, 6))
    msno.matrix(df.sample(min(1000, len(df))))
    plt.title("Matriz de Valores Nulos")
    plt.savefig("/content/images/eda/01_valores_nulos.png", bbox_inches='tight')
    plt.show()

    # Duplicados
    duplicados = df.duplicated().sum()
    print(f"Valores duplicados encontrados: {duplicados}")

    # Tipos de datos
    print("\nTipos de datos por columna:")
    display(df.dtypes)


**Interpretación de Resultados:**
- Se calculó el porcentaje de valores nulos por columna, lo cual es vital para decidir estrategias de imputación o eliminación antes de aplicar modelos de Machine Learning.
- La matriz de valores nulos permite visualizar si hay patrones sistemáticos de datos faltantes.
- Comprobar y eliminar (si es necesario) valores duplicados garantiza que no estemos sobreestimando información.

---

# 4. Análisis de Departamentos

Analizamos el comportamiento a nivel departamental: emisiones, absorciones y balance (emisiones netas).

In [ ]:
if 'df' in locals():
    # Agrupación por departamento
    dept_stats = df.groupby('departamento')[['emisiones_totales', 'abosorciones_totales', 'emisiones_netas']].sum().reset_index()

    # Top 10 emisiones totales
    top10_emisiones = dept_stats.sort_values(by='emisiones_totales', ascending=False).head(10)
    # Top 10 absorciones totales (las absorciones suelen ser negativas, tomamos valor absoluto o los más negativos)
    top10_absorciones = dept_stats.sort_values(by='abosorciones_totales', ascending=True).head(10) # Asumiendo negativo = mayor absorción
    # Top 10 emisiones netas mayores y menores
    top10_netas_mayores = dept_stats.sort_values(by='emisiones_netas', ascending=False).head(10)
    top10_netas_menores = dept_stats.sort_values(by='emisiones_netas', ascending=True).head(10)

    def plot_barh(data, x_col, y_col, title, filename):
        plt.figure(figsize=(10, 6))
        sns.barplot(data=data, x=x_col, y=y_col, palette='viridis')
        plt.title(title)
        plt.xlabel(x_col)
        plt.ylabel(y_col)
        plt.savefig(f"/content/images/eda/{filename}.png", bbox_inches='tight')
        plt.show()

    plot_barh(top10_emisiones, 'emisiones_totales', 'departamento', 'Top 10 Departamentos con Mayores Emisiones', '02_top10_emisiones')
    plot_barh(top10_absorciones, 'abosorciones_totales', 'departamento', 'Top 10 Departamentos con Mayores Absorciones', '03_top10_absorciones')
    plot_barh(top10_netas_mayores, 'emisiones_netas', 'departamento', 'Top 10 Departamentos con Mayores Emisiones Netas', '04_top10_netas_mayores')
    plot_barh(top10_netas_menores, 'emisiones_netas', 'departamento', 'Top 10 Departamentos con Menores Emisiones Netas (Mejor Balance)', '05_top10_netas_menores')


**Interpretación de Resultados:**
- Las gráficas evidencian qué departamentos lideran la generación de emisiones brutas.
- Asimismo, identificamos los departamentos que contribuyen en mayor medida a la captura de carbono (absorciones).
- Analizando las emisiones netas, descubrimos qué departamentos tienen el balance ambiental más desfavorable (mayores emisiones netas) y cuáles el más favorable o cercano a la neutralidad de carbono.

---

# 5. Análisis por Sector Principal

Comparativa del impacto ambiental entre los diferentes sectores económicos (Energía, Procesos Industriales, Agricultura, Silvicultura y Residuos).

In [ ]:
if 'df' in locals():
    sector_stats = df.groupby('sector_principal')[['emisiones_totales', 'abosorciones_totales', 'emisiones_netas']].sum().reset_index()
    sector_melted = pd.melt(sector_stats, id_vars='sector_principal', var_name='Metrica', value_name='Valor')

    plt.figure(figsize=(12, 7))
    sns.barplot(data=sector_melted, x='sector_principal', y='Valor', hue='Metrica')
    plt.title("Emisiones y Absorciones por Sector Principal")
    plt.xticks(rotation=45)
    plt.ylabel("Cantidad (Unidades de GEI)")
    plt.legend(title='Métrica')
    plt.tight_layout()
    plt.savefig("/content/images/eda/06_sectores_comparativa.png")
    plt.show()


**Interpretación de Resultados:**
- Este gráfico de barras agrupadas muestra claramente qué sector económico aporta más a las emisiones totales.
- Podemos ver qué sector es responsable de las absorciones (típicamente Silvicultura).
- En conjunto, identificamos el sector más contaminante y el sector con el mejor balance ambiental neto.

---

# 6. Distribución de Emisiones

Estudio de la distribución de las variables clave: emisiones totales, absorciones y emisiones netas.

In [ ]:
if 'df' in locals():
    fig, axes = plt.subplots(3, 1, figsize=(10, 15))
    vars_to_plot = ['emisiones_totales', 'abosorciones_totales', 'emisiones_netas']

    for i, var in enumerate(vars_to_plot):
        if df[var].notnull().sum() > 0:
            sns.histplot(df[var].dropna(), kde=True, ax=axes[i], bins=50)
            axes[i].set_title(f'Distribución de {var}')
            axes[i].set_xlabel('Valor')
            axes[i].set_ylabel('Frecuencia')
            
    plt.tight_layout()
    plt.savefig("/content/images/eda/07_distribuciones_histogramas.png")
    plt.show()

    # Boxplots
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df[vars_to_plot])
    plt.title("Boxplots de Variables Principales")
    plt.yscale('symlog') # Usando escala logarítmica simétrica para manejar amplios rangos y ceros
    plt.savefig("/content/images/eda/08_distribuciones_boxplots.png")
    plt.show()


**Interpretación de Resultados:**
- Los histogramas y el KDE (estimación de densidad de kernel) nos indican si los datos tienen asimetrías severas (ej. asimetría positiva donde la mayoría de valores son pequeños, pero existen algunos muy grandes).
- Los boxplots muestran la dispersión y sugieren una gran concentración alrededor de cero, además de evidenciar una gran cantidad de valores extremos.

---

# 7. Correlaciones

Análisis de la relación lineal entre los principales gases y los totales calculados.

In [ ]:
if 'df' in locals():
    vars_corr = ['co2', 'ch4', 'n2o', 'co2eq', 'ch4eq', 'n2oeq', 'emisiones_totales', 'abosorciones_totales', 'emisiones_netas']
    # Filtrar solo aquellas columnas que estén en el dataset y sean numéricas
    vars_corr = [v for v in vars_corr if v in df.columns]

    corr_matrix = df[vars_corr].corr()

    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title("Heatmap de Correlaciones")
    plt.savefig("/content/images/eda/09_heatmap_correlaciones.png", bbox_inches='tight')
    plt.show()

    # Ranking de correlaciones con emisiones_netas
    if 'emisiones_netas' in corr_matrix.columns:
        corr_ranking = corr_matrix['emisiones_netas'].sort_values(ascending=False)
        print("Ranking de correlaciones con Emisiones Netas:")
        display(corr_ranking)


**Interpretación de Resultados:**
- El mapa de calor (heatmap) nos permite ver rápidamente qué gases están más estrechamente correlacionados con las emisiones totales y netas.
- Las correlaciones altas y positivas cerca a 1.0 indican relaciones directas fuertes.
- Esto nos ayuda a identificar qué gas (ej. CO2eq o CH4eq) es el principal conductor de la métrica de Emisiones Netas en este dataset.

---

# 8. Análisis de Gases

Comparación de la contribución de los tres principales gases (CO2, CH4, N2O).

In [ ]:
if 'df' in locals():
    # Comparación usando los valores equivalentes para que sean comparables
    gases = ['co2eq', 'ch4eq', 'n2oeq']
    gases = [g for g in gases if g in df.columns]

    if gases:
        totales_gases = df[gases].sum()
        
        plt.figure(figsize=(14, 6))
        
        # Gráfico de barras
        plt.subplot(1, 2, 1)
        sns.barplot(x=totales_gases.index, y=totales_gases.values, palette='Set2')
        plt.title("Contribución Total por Gas (eq)")
        plt.ylabel("Total Emisiones")
        
        # Gráfico circular
        plt.subplot(1, 2, 2)
        plt.pie(totales_gases.values, labels=totales_gases.index, autopct='%1.1f%%', colors=sns.color_palette('Set2'))
        plt.title("Participación Porcentual por Gas")
        
        plt.tight_layout()
        plt.savefig("/content/images/eda/10_analisis_gases.png")
        plt.show()


**Interpretación de Resultados:**
- Analizando los gases en su formato "equivalente", podemos comparar "peras con peras".
- Los gráficos revelan qué gas contribuye en mayor proporción (generalmente el CO2, pero el CH4 puede tener un peso considerable dependiendo del sector agropecuario).
- Se observa claramente el porcentaje dominante de aporte a la contaminación total de GEI.

---

# 9. Outliers

Detección de valores atípicos mediante el Rango Intercuartílico (IQR).

In [ ]:
if 'df' in locals():
    def detect_outliers(df, column):
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
        return len(outliers), lower_bound, upper_bound

    outliers_info = []
    for var in vars_to_plot:
        if var in df.columns:
            n_outliers, lb, ub = detect_outliers(df, var)
            outliers_info.append({'Variable': var, 'Cantidad Outliers': n_outliers, '% del Total': round((n_outliers/len(df))*100, 2)})

    df_outliers = pd.DataFrame(outliers_info)
    display(df_outliers)


**Interpretación de Resultados:**
- La tabla muestra la cantidad absoluta y relativa de outliers en las variables clave.
- Debido a la naturaleza de los datos ambientales (ej., una central térmica masiva vs pequeños comercios), es altamente probable que existan muchos outliers legítimos (no errores de medición).
- **Decisión:** En general, estos valores *no* deben eliminarse sin un análisis de contexto, ya que representan a los "grandes emisores" del país. Se recomienda mantenerlos para los modelos o usar transformaciones logarítmicas si el algoritmo lo requiere.

---

# 10. Insights

Conclusiones generales extraídas de todo el análisis exploratorio.

In [ ]:
if 'df' in locals():
    # Automatización de Insights
    # Calculando respuestas
    dep_mas_contaminante = dept_stats.sort_values(by='emisiones_totales', ascending=False).iloc[0]['departamento']
    dep_mas_absorcion = dept_stats.sort_values(by='abosorciones_totales', ascending=True).iloc[0]['departamento'] # asumiendo valor negativo
    sec_mas_contaminante = sector_stats.sort_values(by='emisiones_totales', ascending=False).iloc[0]['sector_principal']
    sec_mejor_balance = sector_stats.sort_values(by='emisiones_netas', ascending=True).iloc[0]['sector_principal']

    if gases:
        gas_mayor = totales_gases.idxmax()
    else:
        gas_mayor = "No disponible"

    print("### Principales Hallazgos ###")
    print(f"1. Departamento más contaminante (mayores emisiones): {dep_mas_contaminante}")
    print(f"2. Departamento con mejor absorción: {dep_mas_absorcion}")
    print(f"3. Sector más contaminante: {sec_mas_contaminante}")
    print(f"4. Sector con mejor balance ambiental: {sec_mejor_balance}")
    print(f"5. Gas con mayor contribución: {gas_mayor}")
    print(f"6. Variables más relevantes para Machine Learning: departamento, sector_principal, emisiones_netas, y posiblemente el gas dominante.")


**Interpretación de Resultados:**
- Estos insights resumen directamente los hallazgos para los tomadores de decisiones o stakeholders, dando respuestas claras a las preguntas de negocio formuladas al inicio.

---

# 11. Preparación para ML

Clasificación de variables para futuros modelos de Machine Learning (ej. predicción de emisiones netas).

| Variable | Tipo | Uso ML |
| -------- | ---- | ------ |
| departamento | Categórica | Predictora |
| sector_principal | Categórica | Predictora |
| categorias | Categórica | Predictora (alta cardinalidad) |
| co2eq / ch4eq | Numérica | Predictoras adicionales / features |
| emisiones_netas | Numérica | **Objetivo (Target)** |
| abosorciones_totales | Numérica | Predictora |
| emisiones_totales | Numérica | Predictora |


**Interpretación:**
- Se define `emisiones_netas` como posible variable objetivo para un problema de regresión.
- Las variables categóricas requerirán codificación (One-Hot Encoding, Target Encoding).
- Las variables de gases individuales pueden sufrir de multicolinealidad con los totales, un factor a tener en cuenta para la selección de características.

---

# 12. Exportación

Las gráficas se han guardado exitosamente en el directorio `/content/images/eda/`. 
A continuación, proporcionamos el código para comprimir las gráficas y descargarlas directamente desde Colab.

In [ ]:
import shutil
from google.colab import files

print("Archivos generados en /content/images/eda/:")
display(os.listdir('/content/images/eda/'))

# Comprimir la carpeta de imágenes en un archivo ZIP
shutil.make_archive('/content/graficas_eda', 'zip', '/content/images/eda/')

# Descargar automáticamente el archivo ZIP
files.download('/content/graficas_eda.zip')


**Conclusión Final:**
- El EDA está completo y el dataset está listo para las fases de preprocesamiento, Feature Engineering y modelado con Machine Learning. Todas las gráficas están listas para el reporte final y descargadas a tu máquina local.